# 演習3 解答編 ―― `mutex` で守る

> まず `ex03_mutex.ipynb` を自分で解いてから読んでください。

## 3-2 予測クイズの答え

**狭い版のほうが速く、その差はおよそ2倍**（＝スレッド数の分）になります。

広い版は `heavy_work()` を実行している間ずっと鍵を握っているため、
もう1つのスレッドは何もできずに待っています。
2スレッド起動していても、実際には**常に1スレッドしか働いていない**状態です。

一方の狭い版は、鍵の中でやるのはキューを1回触るだけ（数ナノ秒）で、
重い計算は鍵の外＝**2スレッドが本当に同時に**実行できます。

どちらも**答えは同じ**であることも重要です。
鍵の広さは「正しさ」ではなく「速さ」の問題です。
広くしても正しく動いてしまうため、**遅い原因として気づきにくい**という厄介さがあります。

## 発展課題1 の解答 ―― ループ全体をロックする

```cpp
while (true) {
    std::lock_guard<std::mutex> guard(mtx);
    if (q.empty()) return;
    q.pop();
    taken[id]++;
}
```

- **正しく動くか** ⇒ 動きます。合計は必ず 200,000 になります。
  `lock_guard` は `while` の**中**で作られているので、1周ごとに施錠・解錠が繰り返されます
  （`{ }` を書かなくても、`while` の本体そのものがスコープです）。
- **速いか** ⇒ この例ではほとんど変わりません。`q.pop()` も `taken[id]++` も一瞬で終わるからです。
  ただし `taken[id]++` まで鍵の中に入っているのは無駄で、`ex03b.cpp` のように
  鍵の中の処理が重くなった瞬間に効いてきます。

> **さらに一歩**：`lock_guard` を `while` の**外**（ループに入る前）に置くとどうなるでしょうか。
> デッドロックはしませんが、先に鍵を取ったスレッドが 200,000 個**すべてを1人で処理**してしまい、
> もう片方はそれが終わるまで一切動けません（実行すると内訳が `200000 / 0` のようになります）。
> 答えは正しいのに、並列にした意味が完全に失われる例です。
>
> 一方、`while` の外と中の**両方**に `lock_guard` を置くと、同じスレッドが同じ鍵を
> 二重にロックしようとして**本当にデッドロックします**（`std::mutex` は同じスレッドからでも
> 二重にロックできません）。

2つとも確かめてみましょう。まず「`while` の外に置いた版」です。

In [ ]:
%%writefile ans03a.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>

std::queue<int> q;
std::mutex mtx;
long taken[2] = {0, 0};

void worker(int id) {
    std::lock_guard<std::mutex> guard(mtx);     // ← while の「外」で施錠
    while (true) {
        if (q.empty()) return;
        q.pop();
        taken[id]++;
    }
}

int main() {
    for (int i = 0; i < 200000; i++) q.push(i);
    std::thread t1(worker, 0), t2(worker, 1);
    t1.join(); t2.join();
    std::cout << "スレッド0 = " << taken[0] << " / スレッド1 = " << taken[1]
              << " / 合計 = " << taken[0] + taken[1] << "\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans03a.cpp -o ans03a
!for i in 1 2 3; do ./ans03a; done

合計は 200,000 で正しいのに、内訳は **`0 / 200000`**。
先に鍵を取ったスレッドが全部を1人で処理し、もう片方は最後まで何もできていません。
**答えは正しいのに、並列にした意味が完全に失われています。**

次は「外と中の両方に置いた版」＝本物のデッドロックです。
止まったままにならないよう、5秒で強制終了させます。

In [ ]:
%%writefile ans03b.cpp
#include <iostream>
#include <mutex>

std::mutex mtx;

int main() {
    std::cout << "1回目のロック...\n" << std::flush;
    std::lock_guard<std::mutex> g1(mtx);         // while の「外」に相当

    std::cout << "2回目のロック（同じスレッドで同じ鍵）...\n" << std::flush;
    std::lock_guard<std::mutex> g2(mtx);         // while の「中」に相当 → ここで止まる

    std::cout << "ここには絶対に到達しない\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans03b.cpp -o ans03b
!timeout 5 ./ans03b; echo "終了コード=$? （124 なら5秒たっても終わらなかった＝デッドロック）"

「2回目のロック...」を表示したまま、**永久に止まります**。
エラーも出ず、異常終了もせず、ただ固まる ―― これがデッドロックの怖いところです。

## 発展課題2 の解答 ―― `heavy_work` の重さを変える

実測すると、比（広い版 ÷ 狭い版）はおおよそ次のようになります。

```
   WORK        狭い版      広い版       比
--------------------------------------------------
     2,000     約   6 ms   約   30 ms   約 5 倍
    20,000     約  57 ms   約  140 ms   約 2.4 倍
   200,000     約 560 ms   約 1150 ms   約 2.0 倍
```

- **重くするほど、比は 2倍（＝スレッド数）に近づきます。** これが並列化で得られる理論上の上限です。
- **軽くすると、比はむしろ大きくなります。** 広い版では鍵をほぼ握りっぱなしなので、
  もう1つのスレッドが「待たされて眠り、起こされ、また待たされる」を繰り返し、
  その**切り替えのコスト自体**が上乗せされるためです
  （ただし全体が数msだと測定誤差も大きいので、数字は目安として見てください）。

ここから導かれる原則は次のとおりです。

> **鍵の外でできる仕事が多いほど、並列化は効く。**

次のセルで、`heavy_work` の重さを 3 段階に変えて確かめられます。

In [ ]:
%%writefile ans03c.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <chrono>
using namespace std::chrono;

std::mutex mtx;
int WORK = 20000;                    // ← 重さ（ex03b.cpp の 20000 に相当）

long heavy_work(int v) {
    long s = 0;
    for (int i = 0; i < WORK; i++) s += (v + i) % 7;
    return s;
}

long run(bool narrow) {
    std::queue<int> q;
    for (int i = 0; i < 2000; i++) q.push(i);
    long total = 0;
    auto body = [&]() {
        while (true) {
            int v;
            if (narrow) {
                { std::lock_guard<std::mutex> g(mtx); if (q.empty()) return; v = q.front(); q.pop(); }
                long r = heavy_work(v);
                { std::lock_guard<std::mutex> g(mtx); total += r; }
            } else {
                std::lock_guard<std::mutex> g(mtx);
                if (q.empty()) return;
                v = q.front(); q.pop();
                total += heavy_work(v);
            }
        }
    };
    auto t0 = steady_clock::now();
    std::thread t1(body), t2(body);
    t1.join(); t2.join();
    return duration_cast<milliseconds>(steady_clock::now() - t0).count();
}

int main() {
    for (int w : {2000, 20000, 200000}) {
        WORK = w;
        long n = run(true), b = run(false);
        std::cout << "WORK=" << w << "\t狭い版 " << n << " ms\t広い版 " << b << " ms"
                  << "\t(広い/狭い = " << (double)b / n << " 倍)\n";
    }
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans03c.cpp -o ans03c && ./ans03c

`WORK` を大きくするほど「広い/狭い」の比が **2倍（＝スレッド数）に近づき**、
小さくすると **1倍に近づく**（＝差がなくなる）ことが確かめられます。

パイプラインで言えば、キューから取り出したあとの**処理そのものは鍵の外**でやります。
鍵を握るのは、キューに1件入れる/出す一瞬だけ。
だから複数の担当者が本当に同時に動けます。

## 発展課題3 の解答 ―― デッドロック

両方のスレッドが永遠に止まります。

```
スレッドA: 鍵1をロック成功 → 鍵2をロックしたい（でもBが持っている）→ 待つ
スレッドB: 鍵2をロック成功 → 鍵1をロックしたい（でもAが持っている）→ 待つ
```

互いに相手が持っている鍵を待ち、どちらも自分の鍵を離さないので、
永久に動きません。これが**デッドロック**です。

プログラムは異常終了せず、エラーも出さず、ただ**固まります**。
競合と違って「たまたま動く」こともあるため、やはり再現が難しいバグです。

古典的な対策は次の2つです。

- **鍵をかける順番を全スレッドで統一する**（必ず 鍵1 → 鍵2 の順）
- **複数の鍵を同時に取るときは `std::lock` / `std::scoped_lock` を使う**
  （デッドロックしない順序で取ってくれる）

鍵を1つしか使わない設計なら、この問題は起きません。
**鍵は少ないほど安全**、というのも設計上の大事な指針です。

## 発展課題4 の解答 ―― 鍵を握ったまま `notify_one()` を呼ぶ

```cpp
queue_.push(value);
can_pop_.notify_one();      // まだ施錠されている
```

**正しさの問題はありません。** どちらでも正しく動きます。これは効率の話です。

鍵を握ったまま通知すると、次のような無駄が起きることがあります。

1. `notify_one()` で待機中のスレッドBが起こされる
2. B は「待つのをやめる」ために、まず**鍵を取り直さなければならない**
3. しかし鍵はまだAが握っている → **B はもう一度眠る**
4. A がスコープを抜けて解錠 → B がまた起こされて、ようやく鍵を取れる

Bが「起きる → 鍵が取れず眠る → また起きる」と2度手間になります。
これを **hurry up and wait** と呼びます。

したがって、理屈のうえでは

```cpp
{
    std::unique_lock<std::mutex> guard(mtx_);
    ...
    queue_.push(value);
    guard.unlock();          // 先に解錠してから
}
can_pop_.notify_one();       // 通知する
```

のほうが効率的です（`unique_lock` なら途中で解錠できる、という 2-3 の話がここで生きます）。

ただし実際には、最近の実装はこの無駄を減らす工夫をしており、
差はほとんど測定できないことが多いです。
鍵を握ったまま通知する書き方は一般的で、実用上まったく問題ありません。

**この課題の狙い**は「どちらが正しいか」ではなく、
> 同じ動作をするコードでも、**待たせ方**の違いで性能が変わりうる

という視点を持つことです。「正しく動いているのに遅い」原因の多くは、
**鍵の掛け方**にあります。